# 02 Semantic Domain Engineering

This notebook constructs the nine biomedical domain scores used for block-level causal discovery.

Pipeline:
Raw ATTICA balanced cohort → domain-specific feature engineering → generic aggregation → final domain dataframe.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Folder containing the full-cohort notebooks
EXPERIMENT_DIR = Path.cwd().resolve()

# Main project directory
PROJECT_ROOT = EXPERIMENT_DIR.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

RESULTS_DIR = EXPERIMENT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
GRAPHS_DIR = RESULTS_DIR / "graphs"

for directory in [
    TABLES_DIR,
    FIGURES_DIR,
    GRAPHS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


from utils.aggregation import generic_aggregate_engineered
from utils.feature_engineering import (
    engineer_level1_features,
    engineer_level2_features,
    engineer_level3_features,
    engineer_level4_features,
    engineer_level5_features,
    engineer_level6_features,
    engineer_level7_features,
    engineer_level8_features,
    engineer_level9_features,
)

RANDOM_STATE = 42



PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = EXPERIMENT_DIR / "results"



print("Libraries loaded.")

In [ ]:
DATASET_NAME = "attica_full_contrastive_filtered.csv"

df_balanced = pd.read_csv(
    PROCESSED_DIR / DATASET_NAME
)

print("=" * 80)
print("LOADED BALANCED ATTICA DATASET")
print("=" * 80)
print("Shape:", df_balanced.shape)
display(df_balanced.head())





In [ ]:
LABEL = "CVD_mortality10"

if LABEL not in df_balanced.columns:
    raise ValueError("CVD_mortality10 not found. Please check Notebook 01.")

LABELS_SELECTED = df_balanced[LABEL].copy()

print(LABELS_SELECTED.value_counts())

In [ ]:
# ============================================================
# CONSTANT AND NEAR-ZERO-VARIATION AUDIT
# ============================================================

def audit_low_variation(
    df,
    variables,
    domain_name,
    dominant_threshold=0.95,
):
    """
    Identify constant and near-zero-variation variables.

    Constant:
        one or fewer observed unique values.

    Near-zero variation:
        one value represents at least 95% of observed values.
    """
    rows = []

    for variable in variables:

        if variable not in df.columns:
            rows.append({
                "Domain": domain_name,
                "Variable": variable,
                "ObservedCount": 0,
                "UniqueValues": np.nan,
                "DominantShare": np.nan,
                "Status": "missing",
            })
            continue

        series = df[variable].dropna()
        observed_count = len(series)
        unique_values = series.nunique()

        if observed_count == 0:
            dominant_share = np.nan
            status = "all_missing"

        else:
            dominant_share = (
                series.value_counts(normalize=True)
                .iloc[0]
            )

            if unique_values <= 1:
                status = "constant"

            elif dominant_share >= dominant_threshold:
                status = "near_zero_variation"

            else:
                status = "ok"

        rows.append({
            "Domain": domain_name,
            "Variable": variable,
            "ObservedCount": observed_count,
            "UniqueValues": unique_values,
            "DominantShare": dominant_share,
            "Status": status,
        })

    return rows

In [ ]:
LEVEL_ENGINEERS = [
    engineer_level1_features,
    engineer_level2_features,
    engineer_level3_features,
    engineer_level4_features,
    engineer_level5_features,
    engineer_level6_features,
    engineer_level7_features,
    engineer_level8_features,
    engineer_level9_features,
]

In [ ]:
df_balanced.shape

In [ ]:
cfg3 = engineer_level3_features(df_balanced)

behavioral_features = cfg3["features"]
behavioral_variables = cfg3["variables"]

print("Variables:", behavioral_variables)
print("\nDtypes:")
print(behavioral_features[behavioral_variables].dtypes)

print("\nUnique values:")
print(
    behavioral_features[
        behavioral_variables
    ].nunique(dropna=False)
)

print("\nValue counts:")

for variable in behavioral_variables:
    print("\n", variable)

    print(
        behavioral_features[variable]
        .value_counts(dropna=False)
        .sort_index()
    )

print("\nDescriptive statistics:")

display(
    behavioral_features[
        behavioral_variables
    ].describe()
)

print("\nCorrelation:")

display(
    behavioral_features[
        behavioral_variables
    ].corr()
)

In [ ]:
level_results = {}
level_features = {}
level_configs = {}

variation_audit_rows = []

print("=" * 80)
print("RUNNING SEMANTIC FEATURE ENGINEERING + GENERIC AGGREGATION")
print("=" * 80)

for engineer in LEVEL_ENGINEERS:

    cfg = engineer(df_balanced)
    variation_audit_rows.extend(
    audit_low_variation(
        df=cfg["features"],
        variables=cfg["variables"],
        domain_name=cfg["block_name"],
    )
)

    print("\n" + "-" * 80)
    print(cfg["level_name"])
    print("-" * 80)
    print("Block:", cfg["block_name"])
    print("Aggregation variables:", cfg["variables"])

    level_data = generic_aggregate_engineered(
        engineered_df=cfg["features"],
        score_prefix=cfg["score_prefix"],
        variables=cfg["variables"],
        risk_direction=cfg["risk_direction"],
        impute_strategy=cfg["imputation"],
        keep_inputs=True,
        keep_zscores=True
    )

    # Keep engineered trace variables together with score output
    level_data = pd.concat(
        [
            cfg["features"],
            level_data.drop(columns=[
                c for c in cfg["features"].columns 
                if c in level_data.columns
            ], errors="ignore")
        ],
        axis=1
    )

    level_results[cfg["block_name"]] = level_data
    level_features[cfg["block_name"]] = cfg["features"]
    level_configs[cfg["block_name"]] = cfg

    print("Created shape:", level_data.shape)
    print("Score column:", cfg["score_column"])
   
    if cfg["score_column"] in level_data.columns:
        print(level_data[cfg["score_column"]].describe())
        
     
        

In [ ]:

# ============================================================
# EXPORT DOMAIN MEMBERSHIP MANIFEST
# ============================================================

domain_membership_rows = []

for domain_name, features_df in level_features.items():

    for variable in features_df.columns:
        domain_membership_rows.append({
            "Domain": domain_name,
            "EngineeredVariable": variable,
        })

domain_membership_df = pd.DataFrame(
    domain_membership_rows
).sort_values(
    ["Domain", "EngineeredVariable"]
).reset_index(drop=True)

display(domain_membership_df)

domain_membership_df.to_csv(
    TABLES_DIR / "domain_membership_manifest.csv",
    index=False
)

print(
    "Domain membership manifest saved:",
    domain_membership_df.shape
)
domain_membership_summary = (
    domain_membership_df
    .groupby("Domain")
    .agg(
        NumberOfVariables=("EngineeredVariable", "count")
    )
    .reset_index()
)

display(domain_membership_summary)

domain_membership_summary.to_csv(
    TABLES_DIR / "domain_membership_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# EXPORT VARIATION AUDIT
# ============================================================

variation_audit_df = pd.DataFrame(
    variation_audit_rows
)

variation_problems = variation_audit_df[
    variation_audit_df["Status"] != "ok"
].copy()

print("Variation audit summary:")
display(
    variation_audit_df["Status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Count")
)

if variation_problems.empty:
    print(
        "No constant or near-zero-variation aggregation "
        "variables were detected."
    )
else:
    print("Variables requiring review:")
    display(variation_problems)

variation_audit_df.to_csv(
    TABLES_DIR / "engineered_variable_variation_audit.csv",
    index=False,
)

In [ ]:
level1_data = level_results["Demographics"]
level2_data = level_results["SES"]
level3_data = level_results["Behavioral"]
level4_data = level_results["Obesity"]
level5_data = level_results["Metabolic"]
level6_data = level_results["Biomarkers"]
level7_data = level_results["Conditions"]
level8_data = level_results["Treatment"]
level9_data = level_results["Outcomes"]

print("Level outputs assigned.")

In [ ]:
df_blocks = pd.DataFrame(index=df_balanced.index)

for block_name, cfg in level_configs.items():
    score_col = cfg["score_column"]

    if score_col not in level_results[block_name].columns:
        raise ValueError(f"{score_col} not found for {block_name}")

    df_blocks[block_name] = level_results[block_name][score_col]

# Reorder exactly like the main DAG notebook
df_blocks = df_blocks[
    [
        "Demographics",
        "SES",
        "Behavioral",
        "Treatment",
        "Obesity",
        "Metabolic",
        "Biomarkers",
        "Conditions",
        "Outcomes",
    ]
]

print("Final block dataframe shape:", df_blocks.shape)
display(df_blocks.head())

In [ ]:
# ============================================================
# FINAL DOMAIN RANGE AND CONSISTENCY CHECKS
# ============================================================

EXPECTED_DOMAINS = [
    "Demographics",
    "SES",
    "Behavioral",
    "Treatment",
    "Obesity",
    "Metabolic",
    "Biomarkers",
    "Conditions",
    "Outcomes",
]

assert list(df_blocks.columns) == EXPECTED_DOMAINS, (
    "The final domain order is incorrect."
)

assert df_blocks.shape[1] == 9, (
    f"Expected 9 domains, found {df_blocks.shape[1]}."
)

assert not df_blocks.isna().any().any(), (
    "Missing values remain in df_blocks."
)

assert np.isfinite(
    df_blocks.to_numpy(dtype=float)
).all(), (
    "Infinite or non-numeric values exist in df_blocks."
)

outside_unit_range = (
    (df_blocks < -1e-10)
    | (df_blocks > 1 + 1e-10)
)

if outside_unit_range.any().any():

    invalid_locations = (
        outside_unit_range
        .stack()
        .loc[lambda x: x]
        .reset_index()
    )

    invalid_locations.columns = [
        "RowIndex",
        "Domain",
        "Invalid",
    ]

    display(invalid_locations)

    raise ValueError(
        "One or more final domain scores are outside [0, 1]."
    )

print(
    "Final domain dataset passed shape, missingness, "
    "finite-value, and [0, 1] range checks."
)

In [ ]:
print("=" * 80)
print("BLOCK DATAFRAME QUALITY CHECK")
print("=" * 80)

print("\nMissing values:")
display(df_blocks.isna().sum())

print("\nUnique values:")
display(df_blocks.nunique())

print("\nCorrelation matrix:")
display(df_blocks.corr())

In [ ]:
 #============================================================
# VERIFY TREATMENT DIRECTION
# ============================================================

# Treatment is already DietAdherenceScore_Norm:
# higher value = better Mediterranean-diet adherence.
treatment_adherence_score = df_blocks["Treatment"].copy()

print("=" * 90)
print("TREATMENT DIRECTION VERIFIED")
print("=" * 90)
print(
    "Higher Treatment represents better "
    "Mediterranean-diet adherence."
)

display(df_blocks["Treatment"].describe())

assert df_blocks["Treatment"].between(0, 1).all()

In [ ]:
output_path = PROCESSED_DIR / "attica_domain_scores.csv"

df_blocks.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
for block_name, data in level_results.items():
    filename = f"level_{block_name.lower()}_features_scores.csv"
    data.to_csv(PROCESSED_DIR / filename, index=False)
    print("Saved:", filename)

In [ ]:
summary_rows = []

for block_name, cfg in level_configs.items():
    summary_rows.append({
        "Block": block_name,
        "Level": cfg["level_name"],
        "ScorePrefix": cfg["score_prefix"],
        "ScoreColumn": cfg["score_column"],
        "AggregationVariables": ", ".join(cfg["variables"]),
        "Imputation": cfg["imputation"],
    })

domain_summary = pd.DataFrame(summary_rows)

display(domain_summary)

domain_summary.to_csv(TABLES_DIR / "domain_engineering_summary.csv", index=False)

print("Domain summary saved.")

In [ ]:
print("=" * 80)
print("SEMANTIC DOMAIN ENGINEERING COMPLETE")
print("=" * 80)

print("Participants:", df_blocks.shape[0])
print("Domains:", df_blocks.shape[1])

display(df_blocks.describe())

print("\nReady for Notebook 03: Block-Level Causal Discovery")

In [ ]:
FULL_DOMAIN_FILE = (
    PROCESSED_DIR
    / "attica_domain_scores_full.csv"
)

df_blocks.to_csv(
    FULL_DOMAIN_FILE,
    index=False,
)

print("Full-cohort domain matrix saved to:")
print(FULL_DOMAIN_FILE)

print("Shape:", df_blocks.shape)
print(
    "Outcome distribution:",
    df_blocks["Outcomes"]
    .value_counts()
    .sort_index()
    .to_dict(),
)